# PipelineTS 快速入门指南

本教程将带你快速了解 PipelineTS 的核心功能：

1. **加载内置数据集**
2. **使用单个模型进行时间序列预测**
3. **使用 ModelPipeline 自动选择最佳模型**
4. **可视化预测结果**
5. **保存与加载模型**

## 1. 加载内置数据集

PipelineTS 内置了多个时间序列数据集，方便快速验证模型效果。

In [ ]:
from PipelineTS.dataset import (
    LoadElectricDataSets,
    LoadMessagesSentDataSets,
    LoadWebSales,
    LoadSupermarketIncoming
)

# 加载电力数据集
data = LoadElectricDataSets()
print(f"数据形状: {data.shape}")
print(f"列名: {data.columns.tolist()}")
data.head()

## 2. 使用单个模型进行预测

以 LightGBMModel 为例，展示单个模型的基本用法。

In [ ]:
import pandas as pd

# 确保时间列为 datetime 类型
time_col = data.columns[0]
target_col = data.columns[1]
data[time_col] = pd.to_datetime(data[time_col])

print(f"时间列: {time_col}, 目标列: {target_col}")
print(f"时间范围: {data[time_col].min()} ~ {data[time_col].max()}")

In [ ]:
from PipelineTS.ml_model import LightGBMModel

# 初始化模型
model = LightGBMModel(
    time_col=time_col,
    target_col=target_col,
    lags=12,           # 使用过去12个时间步作为特征
    quantile=0.9,      # 90% 预测区间
    n_estimators=200,
    verbose=-1
)

# 训练模型
model.fit(data)
print("模型训练完成！")

In [ ]:
# 预测未来 10 个时间步
result = model.predict(10)
print(f"预测结果形状: {result.shape}")
result

## 3. 可视化预测结果

In [ ]:
from PipelineTS.plot import plot_data_period

# 可视化训练数据与预测结果
plot_data_period(
    data,
    result,
    time_col=time_col,
    target_col=target_col
)

## 4. 使用 ModelPipeline 自动选择最佳模型

ModelPipeline 会自动训练多个模型并选出效果最好的一个。

In [ ]:
from PipelineTS.pipeline import ModelPipeline

# 查看所有可用模型
print("所有可用模型:")
ModelPipeline.list_all_available_models()

In [ ]:
# 创建 Pipeline，使用 'light' 模式（轻量级模型集合，速度较快）
pipeline = ModelPipeline(
    time_col=time_col,
    target_col=target_col,
    lags=12,
    quantile=0.9,
    include_models='light',  # 可选: 'light', 'all', 'nn', 'ml'
    cv=3
)

# 训练所有模型并返回排行榜
leaderboard = pipeline.fit(data)
leaderboard

In [ ]:
# 使用最佳模型预测
best_result = pipeline.predict(10)

# 可视化
plot_data_period(data, best_result, time_col=time_col, target_col=target_col)

## 5. 保存与加载模型

In [ ]:
from PipelineTS.io import save_model, load_model

# 保存 Pipeline
save_model('my_pipeline.zip', pipeline)
print("Pipeline 已保存到 my_pipeline.zip")

# 加载 Pipeline
loaded_pipeline = load_model('my_pipeline.zip')
loaded_result = loaded_pipeline.predict(5)
loaded_result

In [ ]:
# 也可以保存单个模型
best_model = pipeline.get_model()
save_model('best_model.zip', best_model)
print("最佳模型已保存到 best_model.zip")

## 总结

本教程介绍了 PipelineTS 的核心工作流：

| 步骤 | API |
|------|-----|
| 加载数据 | `LoadElectricDataSets()` 等 |
| 单模型预测 | `LightGBMModel(...).fit(data).predict(n)` |
| 自动模型选择 | `ModelPipeline(...).fit(data).predict(n)` |
| 可视化 | `plot_data_period(data1, data2, ...)` |
| 保存/加载 | `save_model(path, model)` / `load_model(path)` |

更多高级功能请参考其他教程。